<a href="https://colab.research.google.com/github/oluwoleadetifa/movie_genre_classification/blob/dev/notebooks/02_text_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path("/content/movie_genre_classification")))
print(sys.path[-1])

/content/movie_genre_classification


In [3]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
!git clone https://github.com/oluwoleadetifa/movie_genre_classification.git

Cloning into 'movie_genre_classification'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (184/184), done.
remote: Compressing objects: 100% (167/167), done.
remote: Total 184 (delta 90), reused 28 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (184/184), 916.49 KiB | 4.24 MiB/s, done.
Resolving deltas: 100% (90/90), done.


In [ ]:
%cd movie_genre_classification

/content/movie_genre_classification


In [ ]:
!pip install pandas numpy scikit-learn torch transformers tqdm joblib

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving IMDB_four_genre_larger_plot_description.csv to IMDB_four_genre_larger_plot_description.csv


In [ ]:
import os
print(os.listdir("/content"))

['.config', 'movie_genre_classification', 'drive', 'sample_data']


In [ ]:
import os
print(os.getcwd())
print(os.listdir("/content/movie_genre_classification"))

/content/movie_genre_classification
['helper.txt', 'requirements-lock.txt', '.git', 'experiments', '.env.example', 'setup.sh', 'docs', 'data', 'README.md', 'IMDB_four_genre_larger_plot_description.csv', 'requirements.txt', 'notebooks', 'environment.yml', 'outputs', '.gitignore', 'src']


In [ ]:
import os
import shutil

os.makedirs("/content/movie_genre_classification/data/raw", exist_ok=True)

shutil.move(
    "/content/movie_genre_classification/IMDB_four_genre_larger_plot_description.csv",
    "/content/movie_genre_classification/data/raw/IMDB_four_genre_larger_plot_description.csv"
)

'/content/movie_genre_classification/data/raw/IMDB_four_genre_larger_plot_description.csv'

In [ ]:
print(os.path.exists("/content/movie_genre_classification/data/raw/IMDB_four_genre_larger_plot_description.csv"))

True


In [ ]:
from src.data.load_data import load_dataset, basic_dataset_check
from src.config import TEXT_COLUMN, LABEL_COLUMN

df = load_dataset()
basic_dataset_check(df)

df.head()

Dataset shape: (1000, 3)

Columns:
['movie_id', 'description', 'genre']

Missing descriptions:
0

Genre distribution:
genre
romance    250
horror     250
comedy     250
action     250
Name: count, dtype: int64


,movie_id,description,genre
0,tt12783454,Elle Evans (Joey King) has finally completed h...,romance
1,tt1798632,A young girl tries to understand how she myste...,horror
2,tt9214832,"In 1800s England, a well meaning but selfish y...",comedy
3,tt8522006,Abby Holland (Kristen Stewart) and Harper Cald...,romance
4,tt21249656,Olga and Maks are 15 years apart. She is a suc...,romance


In [ ]:
print(df.columns.tolist())

['movie_id', 'description', 'genre']


In [ ]:
print(df[[TEXT_COLUMN, LABEL_COLUMN]].head())

                                         description    genre
0  Elle Evans (Joey King) has finally completed h...  romance
1  A young girl tries to understand how she myste...   horror
2  In 1800s England, a well meaning but selfish y...   comedy
3  Abby Holland (Kristen Stewart) and Harper Cald...  romance
4  Olga and Maks are 15 years apart. She is a suc...  romance


In [ ]:
df["text_len"] = (
    df[TEXT_COLUMN]
    .astype(str)
    .apply(lambda x: len(x.split()))
)

df["text_len"].describe()

,text_len
count,1000.000000
mean,331.905000
std,328.521379
min,54.000000
25%,134.000000
50%,230.000000
75%,397.000000
max,2557.000000


In [ ]:
print(df[LABEL_COLUMN].value_counts())

genre
romance    250
horror     250
comedy     250
action     250
Name: count, dtype: int64


In [ ]:
import json
import joblib
import numpy as np

from sklearn.preprocessing import LabelEncoder

from src.config import (
    TEXT_COLUMN,
    LABEL_COLUMN,
    MODELS_DIR,
    METRICS_DIR
)

from src.data.load_data import load_dataset
from src.data.make_splits import create_splits, save_split_ids
from src.data.preprocess_text import preprocess_text_dataframe
from src.features.tfidf_features import fit_transform_tfidf
from src.models.text_models import (
    build_tfidf_baseline_model,
    train_text_model,
    predict_text_model
)
from src.evaluation.metrics import evaluate_multiclass

In [ ]:
import pandas as pd
from src.config import SPLITS_DIR

train_df = pd.read_csv(SPLITS_DIR / "train.csv")
val_df = pd.read_csv(SPLITS_DIR / "val.csv")
test_df = pd.read_csv(SPLITS_DIR / "test.csv")

(600, 4) (200, 4) (200, 4)


In [ ]:
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(train_df[LABEL_COLUMN])
y_val = label_encoder.transform(val_df[LABEL_COLUMN])
y_test = label_encoder.transform(test_df[LABEL_COLUMN])

X_text_train = train_df["clean_text"]
X_text_val = val_df["clean_text"]
X_text_test = test_df["clean_text"]

print(label_encoder.classes_)

['action' 'comedy' 'horror' 'romance']


In [ ]:
import joblib

joblib.dump(
    label_encoder,
    MODELS_DIR / "label_encoder.joblib"
)

print("Saved label encoder.")

Saved label encoder.


In [ ]:
tfidf_outputs = fit_transform_tfidf(
    X_text_train,
    X_text_val,
    X_text_test,
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9
)

X_train = tfidf_outputs["X_train"]
X_val = tfidf_outputs["X_val"]
X_test = tfidf_outputs["X_test"]

print(X_train.shape, X_val.shape, X_test.shape)

(600, 10000) (200, 10000) (200, 10000)


In [ ]:
model = build_tfidf_baseline_model()
model = train_text_model(model, X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


In [ ]:
y_val_pred, y_val_prob = predict_text_model(model, X_val)
val_metrics = evaluate_multiclass(y_val, y_val_pred)

print(val_metrics)

{'accuracy': 0.7, 'macro_f1': 0.6951448234711111, 'macro_precision': 0.6980392156862745, 'macro_recall': 0.7, 'confusion_matrix': [[37, 3, 9, 1], [7, 26, 4, 13], [4, 3, 42, 1], [3, 7, 5, 35]]}


In [ ]:
y_test_pred, y_test_prob = predict_text_model(model, X_test)
test_metrics = evaluate_multiclass(y_test, y_test_pred)

print(test_metrics)

{'accuracy': 0.755, 'macro_f1': 0.7526991508791804, 'macro_precision': 0.7533416875522139, 'macro_recall': 0.7550000000000001, 'confusion_matrix': [[46, 3, 1, 0], [4, 31, 6, 9], [6, 4, 39, 1], [1, 11, 3, 35]]}


In [ ]:
joblib.dump(
    tfidf_outputs["vectorizer"],
    MODELS_DIR / "tfidf_vectorizer.joblib"
)

joblib.dump(
    model,
    MODELS_DIR / "tfidf_text_model.joblib"
)

joblib.dump(
    label_encoder,
    MODELS_DIR / "label_encoder.joblib"
)

np.save(MODELS_DIR / "y_val_text_probs_tfidf.npy", y_val_prob)
np.save(MODELS_DIR / "y_test_text_probs_tfidf.npy", y_test_prob)

results = {
    "val_metrics": val_metrics,
    "test_metrics": test_metrics,
    "classes": label_encoder.classes_.tolist()
}

with open(METRICS_DIR / "tfidf_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("TF-IDF results saved.")

TF-IDF results saved.
